In [26]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    RocCurveDisplay
)



In [38]:
mlflow.end_run()

In [ ]:

data = pd.read_csv(r"../datos/limpio/train.csv")


In [20]:
data.columns

Index(['Unnamed: 0', 'PassengerId', 'Survived', 'Pclass', 'Sex', 'Age',
       'SibSp', 'Parch', 'Fare', 'Embarked'],
      dtype='str')

In [24]:
!python -m mlflow ui

^C


In [29]:
X=data.drop(["Survived"],axis=1)
y=data["Survived"]


In [30]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42)

In [31]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(random_state=42))
])

In [32]:
param_grid = [
    {
        "classifier__solver": ["lbfgs"],
        "classifier__penalty": ["l2"],
        "classifier__C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
        "classifier__max_iter": [500, 1000]
    },
    {
        "classifier__solver": ["liblinear"],
        "classifier__penalty": ["l1", "l2"],
        "classifier__C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
        "classifier__max_iter": [500, 1000]
    }
]

In [33]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    return_train_score=True
)

In [44]:
mlflow.set_experiment("Clasificacion Sobrevivientes2") 
with mlflow.start_run(run_name="LogisticRegression") as run:
    grid_search.fit(X_train, y_train)
    pipeline = grid_search.best_estimator_
    params = grid_search.best_params_
    best = grid_search.best_score_
    mlflow.log_params({
        "best_C": params["classifier__C"],
        "best_penalty": params["classifier__penalty"],
        "best_solver": params["classifier__solver"],
        "best_max_iter":params["classifier__max_iter"],
        "cv_scoring_refit": "f1"
    })
    mlflow.log_metric("cv_best_f1_score", best)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    mse_class = mean_squared_error(y_test, y_pred)
    mse_proba = mean_squared_error(y_test, y_proba)
    mae_class = mean_absolute_error(y_test, y_pred)
    mae_proba = mean_absolute_error(y_test, y_proba)
    r2_class = r2_score(y_test, y_pred)
    r2_proba = r2_score(y_test, y_proba)
    mlflow.log_metrics({
    "test_accuracy": acc,
    "test_precision": prec,
    "test_recall": rec,
    "test_f1_score": f1,
    "test_roc_auc": roc_auc,
    "test_mse_classes": mse_class,
    "test_mse_probabilities_brier": mse_proba,
    "test_mae_classes": mae_class,
    "test_mae_probabilities": mae_proba,
    "test_r2_classes": r2_class,
    "test_r2_probabilities": r2_proba
    })
    

c:\Users\carlo\Documents\Egresado\maestria\ambiente_maestria\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\carlo\Documents\Egresado\maestria\ambiente_maestria\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  war

In [45]:
from sklearn.ensemble import RandomForestClassifier

In [48]:
pipeline_forest = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(random_state=42))
])
param_grid_forest = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 5, 10],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__criterion": ["gini", "entropy"]
}

In [49]:
grid_search_forest = GridSearchCV(
    estimator=pipeline_forest,
    param_grid=param_grid_forest,
    scoring="f1",
    return_train_score=True
)

In [51]:
mlflow.set_experiment("Clasificacion SobrevivientesArbol") 
with mlflow.start_run(run_name="LogisticReRandomForestClassifiergression") as run:
    grid_search_forest.fit(X_train, y_train)
    pipeline = grid_search_forest.best_estimator_
    params = grid_search_forest.best_params_
    best = grid_search_forest.best_score_
    print(params)
    mlflow.log_params({
        "best_n_estimators": params["classifier__n_estimators"],
        "best_max_dept": params["classifier__max_depth"],
        "best_min_samples_split": params["classifier__min_samples_split"],
        "best_min_samples_leafr":params["classifier__min_samples_leaf"],
        "best_classifier__criterion":params["classifier__criterion"],
        "cv_scoring_refit": "f1"
    })
    mlflow.log_metric("cv_best_f1_score", best)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    mse_class = mean_squared_error(y_test, y_pred)
    mse_proba = mean_squared_error(y_test, y_proba)
    mae_class = mean_absolute_error(y_test, y_pred)
    mae_proba = mean_absolute_error(y_test, y_proba)
    r2_class = r2_score(y_test, y_pred)
    r2_proba = r2_score(y_test, y_proba)
    mlflow.log_metrics({
    "test_accuracy": acc,
    "test_precision": prec,
    "test_recall": rec,
    "test_f1_score": f1,
    "test_roc_auc": roc_auc,
    "test_mse_classes": mse_class,
    "test_mse_probabilities_brier": mse_proba,
    "test_mae_classes": mae_class,
    "test_mae_probabilities": mae_proba,
    "test_r2_classes": r2_class,
    "test_r2_probabilities": r2_proba
    })
    

{'classifier__criterion': 'entropy', 'classifier__max_depth': 10, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}
